# 04 — Exploration primes & grecques (Semaine 3, étape 1)

**Objectif.** Réunir les faits pour trancher le fork S3 (README §8bis) : construire les straddles avec
les **primes de la surface** (`impl_premium` de vsurfd) ou **recalculer Black–Scholes** depuis l'IV
(+ taux `zerocd`, dividendes) ?

**Questions :**
1. Que contient exactement `vsurfd` (colonnes disponibles : prime ? strike ? dispersion ?)
2. À quoi ressemblent `zerocd` (taux zéro-coupon) et la table des prix spot (`secprd`) ?
3. **Test décisif** : BS(iv, impl_strike, spot, r, T) reproduit-il `impl_premium` ? Avec quel rendement
   de dividende implicite ?

Date-test : **2024-06-28** (dernier rebalancement de juin 2024, jour du calendrier maître).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from dispersion.data.wrds_client import get_connection

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
db = get_connection()
print("connexion OK")

Loading library list...


Done
connexion OK


In [2]:
# Tables candidates dans optionm : prix spot, taux, forwards
tabs = db.raw_sql("""
SELECT table_name FROM information_schema.tables
WHERE table_schema = 'optionm'
  AND (table_name LIKE 'secprd%%' OR table_name LIKE 'zerocd%%' OR table_name LIKE 'fwdprd%%')
ORDER BY table_name
""")
print(tabs.to_string())

    table_name
0   fwdprd1996
1   fwdprd1997
2   fwdprd1998
3   fwdprd1999
4   fwdprd2000
5   fwdprd2001
6   fwdprd2002
7   fwdprd2003
8   fwdprd2004
9   fwdprd2005
10  fwdprd2006
11  fwdprd2007
12  fwdprd2008
13  fwdprd2009
14  fwdprd2010
15  fwdprd2011
16  fwdprd2012
17  fwdprd2013
18  fwdprd2014
19  fwdprd2015
20  fwdprd2016
21  fwdprd2017
22  fwdprd2018
23  fwdprd2019
24  fwdprd2020
25  fwdprd2021
26  fwdprd2022
27  fwdprd2023
28  fwdprd2024
29  fwdprd2025
30      secprd
31  secprd1996
32  secprd1997
33  secprd1998
34  secprd1999
35  secprd2000
36  secprd2001
37  secprd2002
38  secprd2003
39  secprd2004
40  secprd2005
41  secprd2006
42  secprd2007
43  secprd2008
44  secprd2009
45  secprd2010
46  secprd2011
47  secprd2012
48  secprd2013
49  secprd2014
50  secprd2015
51  secprd2016
52  secprd2017
53  secprd2018
54  secprd2019
55  secprd2020
56  secprd2021
57  secprd2022
58  secprd2023
59  secprd2024
60  secprd2025
61      zerocd


In [3]:
# Schéma complet de la surface vsurfd (année 2024)
cols = db.raw_sql("""
SELECT column_name, data_type FROM information_schema.columns
WHERE table_schema = 'optionm' AND table_name = 'vsurfd2024'
ORDER BY ordinal_position
""")
print(cols.to_string())

       column_name          data_type
0            secid   double precision
1             date               date
2             days   double precision
3            delta   double precision
4  impl_volatility   double precision
5      impl_strike   double precision
6     impl_premium   double precision
7       dispersion   double precision
8          cp_flag  character varying


In [4]:
# Échantillon SPX : 91 jours, ±50 delta, 2024-06-28 — toutes colonnes
spx = db.raw_sql("""
SELECT * FROM optionm.vsurfd2024
WHERE secid = 108105 AND days = 91 AND delta IN (50, -50) AND date = '2024-06-28'
""")
spx

,secid,date,days,delta,impl_volatility,impl_strike,impl_premium,dispersion,cp_flag
0,108105.0,2024-06-28,91.0,-50.0,0.113183,5531.409,127.7991,0.006892,P
1,108105.0,2024-06-28,91.0,50.0,0.121323,5530.623,127.2951,0.007312,C


In [5]:
# Même chose pour la 1re capitalisation de l'univers à ce rebalancement
w = pd.read_parquet(ROOT / "data" / "processed" / "weights.parquet")
top = w[w["rebalance_date"] == "2024-06-28"].nsmallest(1, "rnk")
sec_top = int(top["secid"].iloc[0])
print(top[["permno", "secid", "weight", "rnk"]].to_string(index=False))
comp = db.raw_sql(f"""
SELECT * FROM optionm.vsurfd2024
WHERE secid = {sec_top} AND days = 91 AND delta IN (50, -50) AND date = '2024-06-28'
""")
comp

 permno  secid   weight  rnk
  10107 107525  0.09541    1


,secid,date,days,delta,impl_volatility,impl_strike,impl_premium,dispersion,cp_flag
0,107525.0,2024-06-28,91.0,-50.0,0.211219,452.5007,19.46269,0.006059,P
1,107525.0,2024-06-28,91.0,50.0,0.228016,455.1952,18.95197,0.007014,C


In [6]:
# Courbe zéro-coupon OptionMetrics autour de 91 jours
zc = db.raw_sql("SELECT * FROM optionm.zerocd WHERE date = '2024-06-28' ORDER BY days LIMIT 12")
zc

,date,days,rate
0,2024-06-28,10.0,5.397591
1,2024-06-28,30.0,5.426773
2,2024-06-28,60.0,5.465693
3,2024-06-28,91.0,5.499974
4,2024-06-28,122.0,5.528427
5,2024-06-28,152.0,5.55061
6,2024-06-28,182.0,5.567719
7,2024-06-28,273.0,5.590523
8,2024-06-28,365.0,5.57328
9,2024-06-28,547.0,5.441071


In [7]:
# Prix spot SPX (table secprd, annuelle ou unique selon l'abonnement)
cands = [t for t in tabs["table_name"] if t.startswith("secprd")]
name = "secprd2024" if "secprd2024" in cands else (cands[0] if cands else None)
print("table prix retenue :", name)
spot = db.raw_sql(f"SELECT * FROM optionm.{name} WHERE secid = 108105 AND date = '2024-06-28'")
spot

table prix retenue : secprd2024


,secid,date,low,high,close,volume,return,cfadj,open,cfret,shrout
0,108105.0,2024-06-28,5451.12,5523.64,5460.48,0.0,-0.004084,1.0,5488.48,1.0,0.0


In [8]:
# TEST DÉCISIF : Black-Scholes reproduit-il impl_premium ?
from scipy.stats import norm
from scipy.optimize import brentq

S = abs(float(spot["close"].iloc[0]))
T = 91 / 365.0
r = float(np.interp(91, zc["days"], zc["rate"])) / 100   # % -> décimal, continûment composé
print(f"S = {S:.2f} | r(91j) = {r:.4%} | T = {T:.4f}")

def bs(S, K, sig, T, r, q, cp):
    d1 = (np.log(S / K) + (r - q + 0.5 * sig**2) * T) / (sig * np.sqrt(T))
    d2 = d1 - sig * np.sqrt(T)
    if cp == "C":
        return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)

for label, df in (("SPX", spx), ("top composante", comp)):
    if label == "top composante":
        s_row = db.raw_sql(f"SELECT * FROM optionm.{name} WHERE secid = {sec_top} AND date = '2024-06-28'")
        S_loc = abs(float(s_row["close"].iloc[0]))
    else:
        S_loc = S
    print(f"\n--- {label} (spot {S_loc:.2f}) ---")
    for _, row in df.iterrows():
        K, sig_iv, prem = float(row["impl_strike"]), float(row["impl_volatility"]), float(row["impl_premium"])
        cp = str(row["cp_flag"]).strip()
        p0 = bs(S_loc, K, sig_iv, T, r, 0.0, cp)
        try:
            q_star = brentq(lambda q: bs(S_loc, K, sig_iv, T, r, q, cp) - prem, -0.10, 0.15)
        except ValueError:
            q_star = np.nan
        print(f"{cp}: impl_premium = {prem:9.3f} | BS(q=0) = {p0:9.3f} ({100*(p0/prem-1):+5.1f}%) | q implicite = {q_star:+.2%}")

S = 5460.48 | r(91j) = 5.5000% | T = 0.2493

--- SPX (spot 5460.48) ---
P: impl_premium =   127.799 | BS(q=0) =   120.857 ( -5.4%) | q implicite = +1.04%
C: impl_premium =   127.295 | BS(q=0) =   134.487 ( +5.6%) | q implicite = +1.04%

--- top composante (spot 446.95) ---
P: impl_premium =    19.463 | BS(q=0) =    18.479 ( -5.1%) | q implicite = +1.83%
C: impl_premium =    18.952 | BS(q=0) =    19.329 ( +2.0%) | q implicite = +0.67%


In [9]:
# Couverture HISTORIQUE d'impl_premium sur NOTRE grille figée (91j, ±50δ) — on s'engage sur 29 ans
years = [1996, 2005, 2015]
rows = []
for y in years:
    # SPX, année entière
    a = db.raw_sql(f"""
        SELECT COUNT(*) AS n, COUNT(impl_premium) AS n_prem, COUNT(impl_strike) AS n_strike
        FROM optionm.vsurfd{y}
        WHERE secid = 108105 AND days = 91 AND delta IN (50, -50)
    """)
    # univers point-in-time top-100 au rebalancement de mi-année, le jour du rebalancement
    reb = sorted(d for d in w["rebalance_date"].unique() if pd.Timestamp(d).year == y)[1]
    secs = w.loc[w["rebalance_date"] == reb, "secid"].dropna().astype(int).tolist()
    b = db.raw_sql(f"""
        SELECT COUNT(*) AS n, COUNT(impl_premium) AS n_prem
        FROM optionm.vsurfd{y}
        WHERE secid IN ({', '.join(map(str, secs))}) AND days = 91 AND delta IN (50, -50)
          AND date = '{pd.Timestamp(reb).date()}'
    """)
    rows.append((y, int(a["n"][0]), int(a["n_prem"][0]), int(a["n_strike"][0]),
                 str(pd.Timestamp(reb).date()), len(secs), int(b["n"][0]), int(b["n_prem"][0])))
cov = pd.DataFrame(rows, columns=["année", "SPX lignes", "SPX prem", "SPX strike",
                                  "rebal", "n secids", "univers lignes", "univers prem"])
print("univers lignes attendues = 2 × n secids (call + put)")
cov

univers lignes attendues = 2 × n secids (call + put)


,année,SPX lignes,SPX prem,SPX strike,rebal,n secids,univers lignes,univers prem
0,1996,504,504,504,1996-06-28,100,198,198
1,2005,504,504,504,2005-06-30,100,200,200
2,2015,504,504,504,2015-06-30,100,200,200


## Constats & cadrage par les choix déjà tranchés

**Choix figés, NON réouverts ici** : grille d'extraction = **ATM ±50δ, 91 j** (figée S1 — c'est
exactement la grille échantillonnée ci-dessus) ; ρ_implied en **ATM, MFIV écartée** (fork S2, README
§5) ; **v0 inconditionnel d'abord** et **grecques relatives + test unitaire** (conventions DMV, README
§8bis) ; le fork **coûts** (opprcd vs paramétrique) est distinct et reste ouvert — rien ici ne le
préempte.

**Faits établis par ce notebook :**
1. `vsurfd` porte `impl_premium` + `impl_strike` (+ `dispersion`) sur exactement notre grille figée →
   les primes existent sans changer un seul paramètre de design.
2. **Couverture historique** (cellule ci-dessus) : primes présentes dès 1996, sur SPX comme sur
   l'univers point-in-time.
3. **BS(q=0) rate les primes de ±5 %** (sous-price les puts, sur-price les calls) ; le q implicite
   SPX = **1,04 % identique sur les deux jambes** prouve qu'`impl_premium` est forward-cohérent.
   Recalculer soi-même exigerait le rendement de dividende (indice) et l'exercice américain / dividendes
   discrets (composantes) — lourd et fragile.
4. `zerocd` (taux), `fwdprd` (forwards 1996→2025) et `secprd` (spots) sont disponibles → les grecques
   analytiques sont calculables proprement (IV + strike + forward + taux).

**→ Fork proposé (à valider) : primes = `impl_premium` (ré-extraction, 2 colonnes de plus dans
`get_iv`) ; grecques = BS/Black-76 depuis IV + forward (`fwdprd` ou parité call-put) + `zerocd`.**
Le marquage des positions vieillissantes (91j → 28j : interpolation de surface vs holding-period à la
DMV) = question de design du `DispersionEngine`, posée à l'étape suivante.